In [ ]:
# @title
#Setting de sql y pandas
import sqlite3
import pandas as pd

%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

"""
Dependiendo de la opcion que uses, deberas comentar una opcion u otra para
evitar generar errores y luego cambiarlo en conn = sqlite3.connect(opcion_Elegida)
"""

#Ruta del archivo guardada en drive
#ruta_drive = '/content/drive/MyDrive/Hitos_Skillnest/Hito1/34_SuperTienda_Espanol.db'

#Ruta del archivo subida al entorno, debes subirlo a la raiz del entorno de ejecucion.
ruta = '/content/34_SuperTienda_Espanol.db'

"""
-----------------------------------------------
"""

#Cambiar la variable de ruta segun si estas usando el drive o el archivo subido
#al entorno de ejecucion
conn = sqlite3.connect(ruta) #Se realiza la conexion con la base de datos


#Guardamos la consulta SQL como string, en este caso es para poder conseguir
# el nombre de las tablas presentes en la base de datos.
query = "SELECT name FROM sqlite_master WHERE type='table';"

#ejecutamos la consulta en la base de datos
df_tables =  pd.read_sql_query(query, conn)

print(df_tables) # :D tablas

In [ ]:
#Setting de los Dataframes

df_clientes = pd.read_sql("SELECT * FROM Clientes", conn)

df_productos = pd.read_sql("SELECT * FROM Productos", conn)

df_geografia = pd.read_sql("SELECT * FROM Geografia", conn)

df_cli_ubicacion = pd.read_sql("SELECT * FROM Cliente_Ubicacion", conn)

df_camp = pd.read_sql("SELECT * FROM Campanias", conn)

df_pedidos = pd.read_sql("SELECT * FROM Pedidos", conn)

df_det_pedidos = pd.read_sql("SELECT * FROM Detalle_Pedido", conn)

df_soporte = pd.read_sql("SELECT * FROM Soporte", conn)

Enfoque de clientes

Variables a calcular:

Cuales son los segmentos que mas pedidos realizan, Cuales son las categorias mas pedidas por cada segmento, cuales son los segmentos que producen mayor inversion, los canales preferidos mas utilizados en general, los canales preferidos mas utilizado por cada segmento, la media del puntaje de fidelidad, posible relacion entre segmentos y puntos de fidelidad.


In [ ]:
df_clientes

,ID_Cliente,Nombre_Cliente,Segmento,Fecha_Registro,Canal_Preferido,Puntaje_Fidelidad,Codigo_Interno
0,C0001,Fernanda Díaz,Consumidor,2023-02-18,Web,55,CLI-6635-Z
1,C0002,Valentina Muñoz,Home Office,2023-07-30,Web,58,CLI-2291-Z
2,C0003,Diego Castillo,Corporativo,2024-03-28,App,100,CLI-2139-X
3,C0004,Nicolás Silva,Corporativo,2021-06-13,App,22,CLI-7227-Y
4,C0005,Daniela Rodríguez,Consumidor,2023-01-29,Teléfono,36,CLI-5374-Z
...,...,...,...,...,...,...,...
295,C0296,Catalina Muñoz,Consumidor,2024-08-28,App,78,CLI-1611-X
296,C0297,Sebastián Soto,Consumidor,2023-03-25,Correo,33,CLI-5742-X
297,C0298,Ana Sepúlveda,Home Office,2024-05-19,Web,52,CLI-5662-Y
298,C0299,Catalina Flores,Home Office,2023-10-09,App,74,CLI-8668-Y


In [ ]:
df_cli_ubicacion

In [ ]:
df_geografia

In [ ]:
df_pedidos

,ID_Pedido,ID_Cliente,Fecha_Pedido,Fecha_Envio,Modo_Envio,ID_Campania,Estado_Pedido,Observacion_Interna
0,ORD-2023-00001,C0174,2025-03-31,2025-04-04,Estándar,None,Completado,Cliente frecuente
1,ORD-2023-00002,C0254,2024-03-25,2024-03-25,Segunda Clase,MKT005,Completado,Sin observaciones
2,ORD-2023-00003,C0203,2024-02-20,2024-02-24,Estándar,MKT001,Completado,Revisar descuento aplicado
3,ORD-2023-00004,C0258,2023-10-29,2023-10-31,Primera Clase,None,Enviado,Sin observaciones
4,ORD-2023-00005,C0118,2025-03-30,2025-04-01,Estándar,MKT006,Completado,Verificar dirección
...,...,...,...,...,...,...,...,...
1195,ORD-2025-01196,C0011,2023-12-16,2023-12-22,Primera Clase,MKT008,Completado,Escalado por soporte
1196,ORD-2025-01197,C0215,2023-12-16,2023-12-19,Mismo Día,MKT008,Completado,Verificar dirección
1197,ORD-2025-01198,C0195,2024-06-25,2024-07-01,Mismo Día,MKT007,Completado,Entrega parcial
1198,ORD-2025-01199,C0226,2024-04-12,2024-04-15,Segunda Clase,None,Completado,Sin observaciones


In [ ]:
df_det_pedidos

In [ ]:
df_clientes.describe()

,Puntaje_Fidelidad
count,300.000000
mean,57.143333
std,25.982168
min,10.000000
25%,38.000000
50%,56.000000
75%,80.000000
max,100.000000


In [ ]:
df_soporte

In [ ]:
#unimos por medio de un join las tablas "clientes" y "pedidos".
df_clientes_pedidos = pd.merge(df_clientes,df_pedidos, on="ID_Cliente", how="left")
df_clientes_pedidos.head(1)


In [ ]:
df_clientes_pedidos.info()

In [ ]:
df_clientes_pedidos.head(1).describe(include="all")

In [ ]:
#Identificar que segmento realiza mas pedidos.
seg_pedidos = df_clientes_pedidos.groupby("Segmento")["ID_Pedido"].count().reset_index()
seg_pedidos

In [ ]:
#Canal preferido de los clientes ordenados en forma descendente
canal_pref = df_clientes_pedidos.groupby("Canal_Preferido")["ID_Cliente"].count().reset_index().sort_values("ID_Cliente",ascending=False)
canal_pref


,Canal_Preferido,ID_Cliente
3,Web,317
0,App,316
1,Correo,257
2,Teléfono,257


In [ ]:
#Canal preferido de los clientes segun el segmento.
canal_pref_seg = df_clientes_pedidos.groupby(["Segmento", "Canal_Preferido"])["ID_Cliente"].count().reset_index().sort_values("ID_Cliente",ascending=False)
#Renombramos columnas
canal_pref_seg.columns = ["Segmento","Canal_Preferido", "Cantidad_Clientes"]
#Agrupo por segmento.
canal_mas_usado = canal_pref_seg.groupby("Segmento").head(2)
canal_mas_usado

,Segmento,Canal_Preferido,Cantidad_Clientes
0,Consumidor,App,133
5,Corporativo,Correo,127
6,Corporativo,Teléfono,127
8,Home Office,App,101
3,Consumidor,Web,98
11,Home Office,Web,96


In [ ]:
#Media de puntaje de fidelidad.
media_puntaje = df_clientes_pedidos["Puntaje_Fidelidad"].mean().round(2) #redondeamos para que tenga 2 decimales.
print("puntaje de fidelidad:",media_puntaje)

puntaje de fidelidad: 56.54


In [ ]:
#relacion entre segmentos y puntos de fidelidad.
segmentos_fidelidad = df_clientes_pedidos.groupby("Segmento")["Puntaje_Fidelidad"].mean().reset_index().ro
segmentos_fidelidad

,Segmento,Puntaje_Fidelidad
0,Consumidor,60.511222
1,Corporativo,53.856531
2,Home Office,55.533528
